# GPT Teacher 教程

从零训练一个 3M 参数的中文 GPT 模型，体验完整的训练闭环。

**学习目标**：
1. 理解 GPT 训练的完整流程：分词 → 数据准备 → 训练 → 验收 → 推理
2. 亲手跑通每一步，看到理论变成现实
3. 通过实验理解超参数对模型的影响

## 1. 环境检查

确保所有依赖已安装。

In [ ]:
import sys
print(f"Python: {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"MPS 可用: {torch.backends.mps.is_available()}")

import tokenizers
print(f"Tokenizers: {tokenizers.__version__}")

import matplotlib
print(f"Matplotlib: {matplotlib.__version__}")

print("\n环境检查通过！")

## 2. 分词器

**为什么需要分词器？**

计算机不认识中文。分词器把文字切成一个个 token（词片段），再转成数字。比如「注意力机制」可能被切成 [「注意力」, 「机制」] → [42, 187]。

我们使用 BPE（Byte Pair Encoding），它会自动把经常一起出现的字组合成词。

In [ ]:
import os

# 如果分词器不存在，先构建
if not os.path.exists("tokenizer/tokenizer.json"):
    print("构建分词器...")
    from src.build_tokenizer import build_tokenizer
    build_tokenizer()
else:
    print("分词器已存在")

from src.tokenizer import load_tokenizer
tok = load_tokenizer("hf_tokenizers", "tokenizer/tokenizer.json")
print(f"词表大小: {tok.vocab_size}")

In [ ]:
# 试试分词效果
test_texts = [
    "注意力机制",
    "什么是 Transformer？",
    "蒸馏水和纯水有什么区别？",
]

for text in test_texts:
    ids = tok.encode(text, add_special_tokens=False)
    tokens = [tok.decode([tid]) for tid in ids]
    print(f"原文: {text}")
    print(f"  → {tokens}")
    print(f"  → {ids}")
    print()

## 3. 训练数据

模型通过阅读「问答对」来学习。每条数据包含一个问题（prompt）和一个答案（completion）。

In [ ]:
import json

# 确保训练数据存在
if not os.path.exists("data/train.jsonl"):
    print("生成训练数据...")
    exec(open("data/prepare_data.py").read())

# 查看训练数据
with open("data/train.jsonl", "r", encoding="utf-8") as f:
    lines = [json.loads(l) for l in f if l.strip()]

print(f"训练样本数: {len(lines)}")
print(f"\n前 3 条样本:")
for item in lines[:3]:
    print(f"  Q: {item['prompt']}")
    print(f"  A: {item['completion']}")
    print()

In [ ]:
# 测试集
with open("data/test.jsonl", "r", encoding="utf-8") as f:
    tests = [json.loads(l) for l in f if l.strip()]

print(f"测试样本数: {len(tests)}")
print("\n测试题:")
for i, t in enumerate(tests, 1):
    print(f"  {i}. {t['prompt']}")

## 4. 模型结构

我们使用 Decoder-only Transformer，Llama 风格架构：
- **GQA**（分组查询注意力）：减少 KV Cache，提升推理效率
- **SwiGLU**（门控激活函数）：Llama 风格的前馈网络
- **RMSNorm**：比 LayerNorm 更简洁的归一化
- **RoPE**（旋转位置编码）：注入相对位置信息
- **权重共享**：词嵌入和输出层共享参数

In [ ]:
from src.model import GPT

model = GPT(
    vocab_size=tok.vocab_size,
    n_layer=4,
    n_head=4,
    n_embd=256,
    seq_len=128,
    dropout=0.1,
    n_kv_head=2,
)

total = sum(p.numel() for p in model.parameters())
print(f"参数量: {total:,} ({total/1e6:.2f}M)")
print(f"\n模型结构:")
print(model)

## 5. 训练

训练过程：模型读入问题 → 尝试预测答案 → 对比差距 → 调整参数 → 重复。

默认 5000 步，MPS 约 5 分钟，CPU 约 30-60 分钟。

> 如果不想等待，可以跳过训练，使用预训练模型。

In [ ]:
# 训练模型（取消注释下面的代码来运行）
# from src.train import train
# train(device_arg="auto", max_steps_override=5000)

# 或者快速训练（200步，用于实验）
# train(device_arg="auto", max_steps_override=200)

print("要开始训练，请取消上面的注释。")
print("如果已有预训练模型 (checkpoints/best.pt)，可以直接进行下一步。")

## 6. 验收测试

训练完成后，用验收测试检查模型是否真的学会了。

In [ ]:
import torch
from src.evaluate import load_model, run_test

if os.path.exists("checkpoints/best.pt"):
    model, tok, device = load_model("checkpoints/best.pt")
    passed, total = run_test(model, tok, device)
else:
    print("未找到预训练模型。请先运行训练：取消上面第 5 节的注释并执行。")

## 7. 推理体验

亲手向模型提问，看看它的回答。

In [ ]:
from src.infer import generate

def ask(prompt, temperature=0.0):
    if not os.path.exists("checkpoints/best.pt"):
        print("请先训练模型！")
        return
    model, tok, device = load_model("checkpoints/best.pt")
    response = generate(
        model, tok, prompt,
        max_new_tokens=128, temperature=temperature,
        repetition_penalty=1.5,
        stop_strings=["用户:", "\n用户", "。"],
        device=device,
    )
    print(f"Q: {prompt}")
    print(f"A: {response}")
    print()

# 试试预设问题
ask("什么是注意力机制？")
ask("RoPE 是什么？")
ask("蒸馏水和纯水有什么区别？")

In [ ]:
# 自由提问
your_question = "你好，你是谁？"  # 改成你想问的问题
ask(your_question)

In [ ]:
# Temperature 实验：同一个问题，不同温度
prompt = "什么是注意力机制？"
for temp in [0.0, 0.5, 1.0]:
    print(f"--- Temperature = {temp} ---")
    ask(prompt, temperature=temp)

## 8. 注意力可视化

看看模型在回答问题时，注意力机制关注了哪些词。

In [ ]:
from src.visualize import plot_real_attention, plot_tokenizer

# 真实注意力可视化
if os.path.exists("checkpoints/best.pt"):
    plot_real_attention(
        ckpt_path="checkpoints/best.pt",
        prompt="什么是注意力机制？",
        save_dir="docs",
    )
    print("\n查看 docs/real_attention_*.png")
else:
    print("请先训练模型")

In [ ]:
# 分词器可视化
plot_tokenizer(
    text="注意力机制通过计算相关性分配权重",
    save_dir="docs",
)
print("查看 docs/tokenizer_visualization.png")

## 9. 实验作业

### 入门级
1. 把上面 Temperature 实验中的问题换成其他问题，观察变化
2. 修改 `config/config.yml` 中的 `max_steps` 为 100，重新训练，看看效果如何

### 进阶级
3. 在 `data/prepare_data.py` 中添加一条新的问答（如「什么是深度学习？」），重新生成数据并训练
4. 把 `n_layer` 从 4 改成 2，训练后对比验收结果

### 挑战级
5. 把 `seq_len` 从 128 改成 64，观察哪些问题的回答变差了
6. 尝试把 `n_embd` 从 256 增大到 384，观察模型容量的变化